#### Etape 3.1 : Statistiques descriptives
- Calculer les statistiques par type d'energie, type de batiment et commune
- Identifier les batiments les plus/moins energivores
- Calculer la repartition des consommations par classe energetique DPE
- Analyser l'evolution temporelle (tendances mensuelles, saisonnalite)
- Comparer la consommation theorique (selon DPE) vs reelle

In [60]:
import os
import pandas as pd

INPUT_DIR = os.path.join("../output", "consommations_enrichies.csv")
OUTPUT_DIR = os.path.join("../output", "synthese_dpe.csv")

df_conso = pd.read_csv(INPUT_DIR)

print("colonnes :\n",df_conso.dtypes)
print("nb lignes :\n", len(df_conso))




colonnes :
 Unnamed: 0                            int64
batiment_id                             str
type                                    str
commune                                 str
type_energie                            str
conso_clean                         float64
unite                                   str
date                                    str
hour                                  int64
year                                  int64
month                                 int64
classe_energetique                      str
conso_annuelle                      float64
conso_moyenne_par_occupant_annee    float64
conso_m2                            float64
conso_m2_annee                      float64
cout_horaire                        float64
cout_juor                           float64
cout_mois                           float64
cout_annee                          float64
ipe                                 float64
ecart_ipe                           float64
dtype: object
nb lig

- Calculer les statistiques par type d'energie, type de batiment et commune

In [46]:
#Je prends au m² car ça m'a l'air le plus fiable pour déterminer la conso énergétique (comparer la conso brute de batiments de taille différente n'a pas de sens)

df_stats = df_conso.groupby(["year","commune","type_energie","type"]).agg(
    conso_m2_moyenne=("conso_m2", "sum"),
    conso_m2_min=("conso_m2", "min"),
    conso_m2_max=("conso_m2", "max"),
    conso_m2_std=("conso_m2", "std")
)

print(df_stats)

                                        conso_m2_moyenne  conso_m2_min  \
year commune  type_energie type                                          
2023 Bordeaux eau          ecole               80.283558      0.000091   
                           gymnase             94.263182      0.000953   
                           mairie              68.827880      0.000046   
                           mediatheque         55.569012      0.000560   
                           piscine            694.122361      0.007003   
...                                                  ...           ...   
2024 Toulouse electricite  piscine           1404.440825      0.008831   
              gaz          gymnase           1996.570694      0.012517   
                           mairie            1413.208879      0.000789   
                           mediatheque       4138.965813      0.004902   
                           piscine           2121.557039      0.013252   

                                     

- Identifier les batiments les plus/moins energivores

In [49]:
#par commune
df_conso_excessive = df_conso.groupby(["year","batiment_id", "commune", "type_energie"]) \
    .agg(conso_m2_mean=("conso_m2", "sum")) \
    .sort_values(["commune", "conso_m2_mean"], ascending=[True, False]) \
    .groupby("commune") \
    .head(9)

print(df_conso_excessive)

                                        conso_m2_mean
year batiment_id commune  type_energie               
2024 BAT0043     Bordeaux gaz             4683.686832
2023 BAT0043     Bordeaux gaz             4668.597960
2024 BAT0043     Bordeaux electricite     3125.344599
2023 BAT0043     Bordeaux electricite     3108.444919
2024 BAT0044     Bordeaux gaz             1992.330680
...                                               ...
2023 BAT0036     Toulouse gaz             1560.328315
2024 BAT0036     Toulouse gaz             1556.529478
2023 BAT0037     Toulouse electricite     1408.238646
2024 BAT0037     Toulouse electricite     1404.440825
     BAT0032     Toulouse electricite     1327.973069

[135 rows x 1 columns]


- Calculer la repartition des consommations par classe energetique DPE

In [50]:
df_repartition = df_conso \
    .groupby("classe_energetique", as_index=False) \
    .agg(nb_batiments=("batiment_id", "count")) \
    .sort_values("nb_batiments", ascending=False)


print(df_repartition)

  classe_energetique  nb_batiments
6                  G       2052374
5                  F       1898941
4                  E       1231409
3                  D       1180792
2                  C        769830
1                  B        256597
0                  A        102641


- Analyser l'evolution temporelle (tendances mensuelles, saisonnalite)

In [70]:
# conso au mois
df_mensuel = df_conso \
    .groupby(["month", "type_energie"]) \
    .agg(
        conso_moyenne=("conso_m2", "sum"),
    ).sort_values(["type_energie", "month"])

df_mensuel["variation_mensuelle"] = df_mensuel \
    .groupby("type_energie")["conso_moyenne"] \
    .diff()

print(df_mensuel)


                    conso_moyenne  variation_mensuelle
month type_energie                                    
1     eau             3525.457898                  NaN
2     eau             3251.854458          -273.603440
3     eau             3532.508539           280.654081
4     eau             3391.069326          -141.439213
5     eau             3527.155819           136.086493
6     eau             3412.226684          -114.929135
7     eau             3525.529859           113.303174
8     eau             3533.079520             7.549661
9     eau             3412.875161          -120.204359
10    eau             3521.856799           108.981638
11    eau             3434.643481           -87.213318
12    eau             3514.119746            79.476265
1     electricite    37829.241158                  NaN
2     electricite    34732.830080         -3096.411078
3     electricite    24601.009291        -10131.820789
4     electricite    23711.764824          -889.244467
5     elec

### Ecart DPE 

In [ ]:
df_elec_gaz = df_conso[~df_conso["type_energie"].str.lower().str.contains("eau")].copy()

dpe = {
    "A": 50,
    "B": 90,
    "C": 145,
    "D": 215,
    "E": 290,
    "F": 375,
    "G": 500,
}

df_elec_gaz["seuil_dpe"] = df_elec_gaz["classe_energetique"].map(dpe)

df_comparaison_dpe = df_elec_gaz.groupby(["batiment_id", "classe_energetique", "type_energie"]) \
    .agg(
        conso_reelle_m2=("conso_m2", "sum"),
        seuil_dpe=("seuil_dpe", "first") 
)

df_comparaison_dpe["seuil_depasse"] = (
    df_comparaison_dpe["conso_reelle_m2"] > df_comparaison_dpe["seuil_dpe"]
)

print(df_comparaison_dpe)
#TODO résultats très douteux




                                             conso_reelle_m2  seuil_dpe  \
batiment_id classe_energetique type_energie                               
BAT0001     E                  electricite       1252.019034        290   
                               gaz               1867.471973        290   
BAT0002     C                  electricite        865.683330        145   
                               gaz               1300.152033        145   
BAT0003     D                  electricite        960.940000        215   
...                                                      ...        ...   
BAT0144     E                  gaz               6108.258301        290   
BAT0145     C                  electricite        984.424659        145   
                               gaz               1406.614037        145   
BAT0146     F                  electricite       5332.180360        375   
                               gaz               7984.660974        375   

                        

In [63]:
print("export synthèse")
df_comparaison_dpe.to_csv(OUTPUT_DIR)

export synthèse
